# Femmestral — Text Misinformation Detection Pipeline

This notebook implements the full Femmestral pipeline for detecting women's health misinformation:
- **Fine-tuned Mistral 7B** (`elenaajayi/femmestral-mistral-7b-v2`) for structured verdict generation
- **Gemini** for claim extraction + **Google Search grounding** for citations
- **PubMed + Semantic Scholar** RAG for biomedical evidence
- **NVIDIA Nemotron** content safety (input + output filtering)
- **Weave** tracing for full observability

Run in **Google Colab with a T4 GPU runtime** (Runtime > Change runtime type > T4 GPU).

In [1]:
# Install dependencies (run once in Colab)
!pip -q install -U requests google-genai pandas weave wandb
!pip -q install -U transformers peft accelerate bitsandbytes
!pip -q install -U torch --index-url https://download.pytorch.org/whl/cu118

# Clone the repo to get safety filters + RAG modules
!git clone https://github.com/elenaajayi/femmestral.git /content/femmestral 2>/dev/null || echo "Already cloned"
import sys
if '/content/femmestral' not in sys.path:
    sys.path.insert(0, '/content/femmestral')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.2/732.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.0/893.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 50.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
# Load API keys from Colab Secrets (add via the key icon in the left sidebar)
# Required: GEMINI_API_KEY, NVIDIA_API_KEY, WANDB_API_KEY, HF_TOKEN, NCBI_API_KEY
# Optional: MISTRAL_API_KEY (only needed if using Mistral API for routing)
import os

try:
    from google.colab import userdata
    for key in ["MISTRAL_API_KEY", "GEMINI_API_KEY", "NVIDIA_API_KEY",
                "WANDB_API_KEY", "HF_TOKEN", "NCBI_API_KEY"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
    print("Keys loaded from Colab Secrets")
except ImportError:
    print("Not in Colab -- loading from .env")
    from dotenv import load_dotenv
    load_dotenv()

Keys loaded from Colab Secrets


In [3]:
import os, re, json, time, hashlib
from pathlib import Path
import torch
import weave
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Initialize Weave tracing
weave.init(os.getenv("WANDB_PROJECT", "femmestral-text-pipeline"))

# ====== Load fine-tuned model from HuggingFace ======
HF_REPO = "elenaajayi/femmestral-mistral-7b-v2"
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
hf_token = os.environ.get("HF_TOKEN")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print("Loading fine-tuned model:", HF_REPO)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
femmestral_model = PeftModel.from_pretrained(base_model, HF_REPO, token=hf_token)
femmestral_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)
if femmestral_tokenizer.pad_token is None:
    femmestral_tokenizer.pad_token = femmestral_tokenizer.eos_token
print("Fine-tuned model loaded on:", femmestral_model.device)

# ====== Gemini model (for grounding + claim extraction) ======
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# ====== Mistral API model (for routing only -- lightweight classification) ======
MISTRAL_MODEL = os.getenv("MISTRAL_MODEL", "mistral-small-latest")

# ====== Import safety filters + RAG from repo ======
from src.safety.nemotron import filter_input, filter_output
from src.rag.pubmed import search_pubmed
from src.rag.semantic_scholar import search_semantic_scholar

# ====== Paths ======
INPUT_DIR  = Path("inputs")
OUTPUT_DIR = Path("outputs")
CACHE_DIR  = Path("cache")
for d in [INPUT_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(exist_ok=True)

print("Setup complete -- fine-tuned model + Nemotron safety + PubMed RAG ready")
print("Weave traces: https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/weave")

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/weave


Loading fine-tuned model: elenaajayi/femmestral-mistral-7b-v2


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Fine-tuned model loaded on: cuda:0
Setup complete -- fine-tuned model + Nemotron safety + PubMed RAG ready
Weave traces: https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/weave


## Gemini grounding + strict JSON (fix for `Tool use with response_mime_type='application/json'`)

You hit this SDK limitation:

- When **tools** (Google Search grounding) are enabled, Gemini **cannot** return `response_mime_type="application/json"` in the **same** call.
- So we do **two steps**:
  1) **Grounded search (tools ON)** → returns normal **text**
  2) **Format to STRICT JSON (tools OFF)** → returns **JSON**

The next cells implement this with **debug prints** so you can see *inputs* and *outputs*.


In [9]:
# Cell G1: Imports + Gemini client setup (one-time)
# What this does:
# - Imports the Gemini SDK pieces we need for grounding
# - Creates `gclient` once so all later cells can use it

import os
import json
import re

from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

#  IMPORTANT: Set your key in the environment (DO NOT hardcode in shared notebooks)
# Example (temporary): os.environ["GEMINI_API_KEY"] = "YOUR_KEY"

assert "GEMINI_API_KEY" in os.environ and os.environ["GEMINI_API_KEY"].strip(),     "Missing GEMINI_API_KEY env var. Set it with: os.environ['GEMINI_API_KEY'] = '...'."

# Create the global Gemini client used everywhere
gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Keep GEMINI_MODEL consistent:
# - If GEMINI_MODEL was defined earlier in the notebook, reuse it
# - Otherwise, default to env var or a sensible default
try:
    GEMINI_MODEL
except NameError:
    GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

print(" Gemini client ready")
print("Model:", GEMINI_MODEL)

 Gemini client ready
Model: gemini-2.5-flash


In [10]:
# Cell G2: Why the error happened (simple demo)
# You previously used:
#   tools=[Tool(google_search=GoogleSearch())]
#   response_mime_type="application/json"
# in the same call, which triggers:
#   400 INVALID_ARGUMENT: Tool use with a response mime type 'application/json' is unsupported

print(" Fix = two-step pipeline")
print("Step A: tools ON (GoogleSearch) -> grounded TEXT")
print("Step B: tools OFF -> convert grounded TEXT into STRICT JSON")

 Fix = two-step pipeline
Step A: tools ON (GoogleSearch) -> grounded TEXT
Step B: tools OFF -> convert grounded TEXT into STRICT JSON


In [11]:
# Cell G3: Verification output schema (matches the rest of this notebook)
# This is the JSON shape we want AFTER grounding.
# NOTE: This schema is used in Step B (tools OFF), where JSON mime type IS allowed.

VERIFY_SCHEMA = {
  "type": "object",
  "properties": {
    "stance": {"type": "string", "enum": ["supported", "contradicted", "unclear"]},
    "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
    "rationale": {"type": "string"},
    "citations": {"type": "array", "items": {"type": "string"}}
  },
  "required": ["stance", "confidence", "rationale", "citations"]
}

print(" VERIFY_SCHEMA loaded")
print(json.dumps(VERIFY_SCHEMA, indent=2)[:400] + "...")

 VERIFY_SCHEMA loaded
{
  "type": "object",
  "properties": {
    "stance": {
      "type": "string",
      "enum": [
        "supported",
        "contradicted",
        "unclear"
      ]
    },
    "confidence": {
      "type": "string",
      "enum": [
        "low",
        "medium",
        "high"
      ]
    },
    "rationale": {
      "type": "string"
    },
    "citations": {
      "type": "array",
      "items...


In [12]:
# Cell G4: Helper to safely parse JSON even if the model adds extra text
# (Usually Step B returns clean JSON, but this avoids random demo failures.)

def _extract_json_from_text(s: str) -> dict:
    s = (s or "").strip()

    # Case 1: already pure JSON
    if s.startswith("{") and s.endswith("}"):
        return json.loads(s)

    # Case 2: find first JSON object inside text
    m = re.search(r"\{.*\}", s, re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(0))

print(" JSON extraction helper ready")

 JSON extraction helper ready


In [13]:
# Cell G5: Grounded claim verification (TWO STEPS) with DEBUG PRINTS
# Step A (tools ON): grounded text
# Step B (tools OFF): convert to STRICT JSON per VERIFY_SCHEMA

@weave.op()
def text_pipeline_verify_claim(claim_text: str, debug: bool = True) -> dict:
    # -----------------------
    # Step A: Grounded search
    # -----------------------
    grounded_prompt = f"""Verify the claim below using Google Search grounding.

Claim:
{claim_text}

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
""".strip()

    if debug:
        print("\n" + "="*90)
        print("STEP A INPUT (grounded_prompt):")
        print(grounded_prompt[:1200] + ("..." if len(grounded_prompt) > 1200 else ""))
        print("="*90)

    grounded_resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=grounded_prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            tools=[Tool(google_search=GoogleSearch())],
        ),
    )

    grounded_text = (grounded_resp.text or "").strip()

    if debug:
        print("STEP A OUTPUT (grounded_text):")
        print(grounded_text[:1600] + ("..." if len(grounded_text) > 1600 else ""))
        print("="*90)

    # -----------------------
    # Step B: Format as JSON
    # -----------------------
    json_prompt = f"""Convert the grounded evidence into STRICT JSON that matches this schema:
{json.dumps(VERIFY_SCHEMA, indent=2)}

Rules:
- Output ONLY valid JSON. No markdown. No commentary.
- stance must be: supported, contradicted, or unclear
- confidence must be: low, medium, or high
- citations must be a list of URLs (strings), 2-6 items when possible

Grounded evidence:
{grounded_text}
""".strip()

    if debug:
        print("STEP B INPUT (json_prompt):")
        print(json_prompt[:1400] + ("..." if len(json_prompt) > 1400 else ""))
        print("="*90)

    json_resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=json_prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema=VERIFY_SCHEMA,
        ),
    )

    raw = (json_resp.text or "").strip()

    if debug:
        print("STEP B OUTPUT RAW (should be JSON):")
        print(raw[:1600] + ("..." if len(raw) > 1600 else ""))
        print("="*90)

    try:
        out = json.loads(raw)
    except Exception:
        out = _extract_json_from_text(raw)

    if debug:
        print("PARSED JSON OUTPUT (dict):")
        print(json.dumps(out, indent=2)[:1600] + ("..." if len(json.dumps(out)) > 1600 else ""))
        print("="*90)

    return out

# Keep the old name as alias so downstream cells still work
verify_claim_grounded = text_pipeline_verify_claim

print("text_pipeline_verify_claim() ready — traced in Weave")

text_pipeline_verify_claim() ready — traced in Weave


In [14]:
# Cell G6: Quick sanity test (optional)
# Run this AFTER you have a claim string you want to test.

test_claim = "HPV vaccines cause infertility in women."
try:
    test_out = verify_claim_grounded(test_claim, debug=True)
    test_out
except Exception as e:
    print(" Test failed:", repr(e))


STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
HPV vaccines cause infertility in women.

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd131-0e68-776c-9dda-c5e867788db7


STEP A OUTPUT (grounded_text):
**Stance Suggestion:** Contradicted

**Evidence-Based Assessment:**

The claim that HPV vaccines cause infertility in women is contradicted by extensive scientific evidence from leading global health organizations and numerous studies. There is no credible evidence to support an association between HPV vaccination and infertility or primary ovarian insufficiency (POI).

*   Multiple large-scale studies, including population-based cohort studies, have found no increased risk of infertility among women who received the HPV vaccine.
*   Leading health authorities such as the Centers for Disease Control and Prevention (CDC), the World Health Organization (WHO), the American College of Obstetricians and Gynecologists (ACOG), and the National Health Service (NHS) consistently affirm that HPV vaccines are safe and do not negatively impact fertility.
*   Experts state there is no plausible biological mechanism by which the HPV vaccine could affect fertility, as i

## Stage 1 — Create (synthetic) WhatsApp forward
You can replace this with any file coming from your frontend later.


In [15]:
dubai_msg = """ Forwarded as received

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
 Drink hot water + turmeric
 Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know
"""

msg_path = INPUT_DIR / "dubai_forward_001.txt"
msg_path.write_text(dubai_msg, encoding="utf-8")
print(" Wrote:", msg_path)
print(dubai_msg)


 Wrote: inputs/dubai_forward_001.txt
 Forwarded as received 

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
 Drink hot water + turmeric
 Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know 



## Stage 2 — Load raw content

In [16]:
raw_text = msg_path.read_text(encoding="utf-8", errors="ignore")
print("Loaded chars:", len(raw_text))
print(raw_text)


Loaded chars: 312
 Forwarded as received 

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
 Drink hot water + turmeric
 Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know 



## Stage 3 — Canonicalize text (stable for hashing/dedup)

In [17]:
def canonicalize_text(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"forwarded as received|fwd as received|copied message", "", t)
    t = re.sub(r"[!]{2,}", "!", t)
    t = re.sub(r"[?]{2,}", "?", t)
    return t.strip()

canon_text = canonicalize_text(raw_text)
print("Canonical chars:", len(canon_text))
print(canon_text)


Canonical chars: 281
doctors in dubai are warning women: if you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility. instead: drink hot water + turmeric avoid painkillers for 3 days this is from a “hospital circular”. share this with every woman you know


## Stage 4 — Hashing (dedupe key)
We hash both raw and canonical text. Canonical hash is used to detect reprocessing of the *same* message with different formatting.


In [18]:
def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

canonical_sha256 = sha256_hex(canon_text)
raw_sha256 = sha256_hex(raw_text)

print("canonical_sha256:", canonical_sha256)
print("raw_sha256      :", raw_sha256)


canonical_sha256: 1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362
raw_sha256      : e9316364a1c77aa06461989b96fc3177d969b9fdd5592577a55cb0ac605d5875


## Stage 4.5 — Cache check (avoid reprocessing)
If we already produced a Stage 8 verdict for this canonical hash, we can skip.


In [19]:
stage8_path = OUTPUT_DIR / f"{canonical_sha256}_stage8_verdict.json"
if stage8_path.exists():
    print(" Already processed. Found:", stage8_path)
    existing = json.loads(stage8_path.read_text(encoding="utf-8"))
    print("Final verdict:", existing.get("final"))
else:
    print(" Not processed before. Continue.")


 Not processed before. Continue.


## Stage 5 — Mistral routing/classification (no SDK; raw HTTPS)
This keeps the **Mistral theme** without pydantic dependency issues.


In [20]:
import requests

ROUTER_SYSTEM = """You are a routing classifier for a misinformation detection pipeline focused on women.
Return ONLY valid JSON. No markdown.

Schema:
{
  "domain": "health|safety|legal|finance|bias|unknown",
  "risk_level": "low|medium|high",
  "route": "fast_check|deep_verify",
  "reasons": ["..."],
  "triggers": ["..."]
}

Rules:
- high risk if medical/legal/safety advice, infertility/cure claims, panic forwards, or instructions to share
- deep_verify if risk is medium or high, else fast_check
""".strip()

@weave.op()
def mistral_chat_json(system_prompt: str, user_text: str, model: str, max_tokens: int = 400) -> dict:
    api_key = os.getenv("MISTRAL_API_KEY")
    if not api_key:
        raise ValueError("Missing MISTRAL_API_KEY. Set it at the top of the notebook.")

    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
    }
    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Mistral API error {r.status_code}: {r.text}")

    data = r.json()
    raw = data["choices"][0]["message"]["content"].strip()

    first, last = raw.find("{"), raw.rfind("}")
    if first == -1 or last == -1:
        raise ValueError("Mistral did not return JSON:\n" + raw)

    return json.loads(raw[first:last+1])

@weave.op()
def text_pipeline_route(text: str) -> dict:
    return mistral_chat_json(ROUTER_SYSTEM, text.strip(), model=MISTRAL_MODEL, max_tokens=400)

routing = text_pipeline_route(raw_text)
routing

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd146-7351-7bfc-bc7b-b1e6f55c053e


{'domain': 'health',
 'risk_level': 'high',
 'route': 'deep_verify',
 'reasons': ['Medical advice regarding painkillers and infertility',
  'Instructions to share the message widely'],
 'triggers': ['Medical claims', 'Infertility claims', 'Panic forwards']}

## Stage 5 output — Save routing artifact

In [21]:
result_stage5 = {
    "pipeline_version": "v1",
    "trace_id": f"local-{int(time.time())}",
    "received_at": time.time(),
    "asset": {"path": str(msg_path), "type": "text"},
    "hashes": {"canonical_sha256": canonical_sha256, "raw_sha256": raw_sha256},
    "content": {"raw_preview": raw_text[:300], "canonical_text": canon_text},
    "routing_model": MISTRAL_MODEL,
    "routing": routing,
    "domain": routing.get("domain"),
    "risk_level": routing.get("risk_level"),
    "route": routing.get("route"),
}

stage5_path = OUTPUT_DIR / f"{canonical_sha256}_stage5.json"
stage5_path.write_text(json.dumps(result_stage5, indent=2), encoding="utf-8")
print(" Saved:", stage5_path)


 Saved: outputs/1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage5.json


## Stage 6 — Claim extraction (Gemini structured JSON)
We extract atomic, checkable claims to verify individually.


In [22]:
import os
from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

# Gemini client (expects GEMINI_API_KEY set in env)
if not os.getenv('GEMINI_API_KEY'):
    raise EnvironmentError(
        'Missing GEMINI_API_KEY. Set it as an environment variable before running.'
    )

gclient = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print(' Gemini ready (gclient initialized)')


 Gemini ready (gclient initialized)


In [23]:
CLAIM_SCHEMA = {
  "type": "object",
  "properties": {
    "language": {"type": "string"},
    "claims": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "claim_id": {"type": "string"},
          "text": {"type": "string"},
          "claim_type": {
            "type": "string",
            "enum": ["medical_causal","medical_advice","authority_source","statistical","other"]
          },
          "check_priority": {"type": "string", "enum": ["low","medium","high"]},
          "entities": {"type": "array", "items": {"type": "string"}},
          "implied_advice": {"type": ["string", "null"]}
        },
        "required": ["claim_id","text","claim_type","check_priority","entities"]
      }
    }
  },
  "required": ["language","claims"]
}


In [24]:
@weave.op()
def text_pipeline_extract_claims(text: str) -> dict:
    prompt = f"""You extract checkable claims from WhatsApp/Reddit messages related to women.

Return STRICT JSON that matches the provided schema.
Rules:
- Split into 2-6 atomic claims
- Include authority references (e.g., 'doctors said', 'hospital circular') as authority_source
- Include advice (do/don't) as medical_advice with implied_advice

Message:
{text}
""".strip()

    resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema=CLAIM_SCHEMA,
        ),
    )
    return json.loads(resp.text)

# Keep old name as alias
extract_claims_with_gemini = text_pipeline_extract_claims

claims_out = text_pipeline_extract_claims(raw_text)
claims_out

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd146-d104-796c-85c4-f76afa977fc1


{'language': 'en',
 'claims': [{'claim_id': 'claim_1',
   'text': 'Doctors in Dubai are warning women:',
   'claim_type': 'authority_source',
   'check_priority': 'high',
   'entities': ['Doctors in Dubai']},
  {'claim_id': 'claim_2',
   'text': 'If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.',
   'claim_type': 'medical_causal',
   'check_priority': 'high',
   'entities': ['painkillers',
    'ibuprofen',
    'period',
    'blood flow',
    'infertility']},
  {'claim_id': 'claim_3',
   'text': 'Drink hot water + turmeric',
   'claim_type': 'medical_advice',
   'check_priority': 'medium',
   'entities': ['hot water', 'turmeric'],
   'implied_advice': 'Drink hot water with turmeric to alleviate period symptoms.'},
  {'claim_id': 'claim_4',
   'text': 'Avoid painkillers for 3 days',
   'claim_type': 'medical_advice',
   'check_priority': 'high',
   'entities': ['painkillers'],
   'implied_advice': 'Do not take painkillers for 3 d

In [25]:
def validate_claims(obj: dict) -> None:
    assert isinstance(obj, dict), "Output is not a dict"
    assert "language" in obj and isinstance(obj["language"], str), "Missing/invalid language"
    assert "claims" in obj and isinstance(obj["claims"], list), "Missing/invalid claims list"
    for c in obj["claims"]:
        for k in ["claim_id","text","claim_type","check_priority","entities"]:
            assert k in c, f"Claim missing {k}"
        assert c["claim_type"] in ["medical_causal","medical_advice","authority_source","statistical","other"]
        assert c["check_priority"] in ["low","medium","high"]
        assert isinstance(c["entities"], list)

validate_claims(claims_out)
print(" Claims validated. Count =", len(claims_out["claims"]))


 Claims validated. Count = 6


In [26]:
for c in claims_out["claims"]:
    print(f"{c['claim_id']} | {c['claim_type']} | priority={c['check_priority']}")
    print("  ", c["text"])
    if c.get("implied_advice"):
        print("  advice:", c["implied_advice"])
    if c.get("entities"):
        print("  entities:", ", ".join(c["entities"]))
    print()


claim_1 | authority_source | priority=high
   Doctors in Dubai are warning women:
  entities: Doctors in Dubai

claim_2 | medical_causal | priority=high
   If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.
  entities: painkillers, ibuprofen, period, blood flow, infertility

claim_3 | medical_advice | priority=medium
   Drink hot water + turmeric
  advice: Drink hot water with turmeric to alleviate period symptoms.
  entities: hot water, turmeric

claim_4 | medical_advice | priority=high
   Avoid painkillers for 3 days
  advice: Do not take painkillers for 3 days during your period.
  entities: painkillers

claim_5 | authority_source | priority=high
   This is from a “hospital circular”.
  entities: hospital circular

claim_6 | other | priority=low
   Share this with every woman you know



In [27]:
stage6_path = OUTPUT_DIR / f"{canonical_sha256}_stage6_claims.json"
stage6_path.write_text(json.dumps(claims_out, indent=2), encoding="utf-8")
print(" Saved:", stage6_path)


 Saved: outputs/1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage6_claims.json


## Stage 7–8 — Grounded verification (Gemini + Google Search)
Each claim is checked with grounded web evidence and citations.


In [28]:
from google.genai.types import Tool, GoogleSearch

TRUSTED_DOMAINS = [
    "who.int", "cdc.gov", "nih.gov", "ncbi.nlm.nih.gov", "medlineplus.gov",
    "nhs.uk", "acog.org", "aafp.org", "cochrane.org",
    "mayoclinic.org", "clevelandclinic.org", "hopkinsmedicine.org"
]

from urllib.parse import urlparse

def domain_of(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def is_trusted(url: str) -> bool:
    d = domain_of(url)
    return any(d == td or d.endswith("." + td) for td in TRUSTED_DOMAINS)


In [29]:
# (Deprecated) Old one-shot tool+JSON approach removed
# Gemini SDK does NOT support using GoogleSearch tool AND response_mime_type="application/json"
# in the same call.
#
# Use verify_claim_grounded() defined earlier in cells G1–G5.

print(" Using the two-step verify_claim_grounded() defined above (tools ON -> text, tools OFF -> JSON).")

 Using the two-step verify_claim_grounded() defined above (tools ON -> text, tools OFF -> JSON).


## Fine-tuned Model Verdict (Femmestral 7B + PubMed RAG + Nemotron Safety)

This is the core differentiator: the fine-tuned Mistral 7B model generates structured verdicts
grounded in PubMed evidence, with Nemotron content safety filtering on both input and output.

The Gemini grounding above provides supplementary web evidence. The fine-tuned model
provides the authoritative verdict in the format it was specifically trained to produce.

In [30]:
FEMMESTRAL_SYSTEM = """You are Femmestral, a women's health misinformation fact-checker.
You will be given a health claim and relevant medical evidence from PubMed.
Respond ONLY in this exact format:

**Verdict: [False/Misleading/Partially True/True]**
**Confidence: [High/Medium/Low]** | **Evidence: [A/B/C]** | **Severity: [High/Medium/Low]**

[2-3 sentence explanation grounded in the provided evidence]

**Source:** [cite the most relevant source]

**Share this correction:**
[1-2 sentence WhatsApp-friendly correction]"""


def _retrieve_evidence(claim_text: str) -> list:
    """Retrieve and merge evidence from PubMed + Semantic Scholar."""
    pubmed = search_pubmed(claim_text, max_results=3)
    semantic = search_semantic_scholar(claim_text, max_results=3)
    combined = pubmed + semantic
    seen = set()
    deduped = []
    for r in combined:
        if r["url"] not in seen:
            seen.add(r["url"])
            deduped.append(r)
    grade_order = {"A": 0, "B": 1, "C": 2}
    deduped.sort(key=lambda r: grade_order.get(r["evidence_grade"], 2))
    return deduped[:5]


def _build_evidence_prompt(claim: str, evidence: list) -> str:
    evidence_block = ""
    for i, paper in enumerate(evidence, 1):
        evidence_block += (
            f"\n[{i}] {paper['title']}\n"
            f"    {paper['abstract']}\n"
            f"    Source: {paper['source']} | Grade: {paper['evidence_grade']}\n"
        )
    if evidence_block:
        return (
            f"Claim to fact-check: {claim}\n\n"
            f"Relevant medical evidence:\n{evidence_block}\n"
            f"Based on the evidence above, fact-check this claim."
        )
    return f"Claim to fact-check: {claim}\n\nFact-check this claim using your medical knowledge."


@weave.op()
def femmestral_fact_check(claim_text: str, debug: bool = True) -> dict:
    """Full pipeline: Nemotron safety -> PubMed RAG -> fine-tuned model -> Nemotron safety."""

    # 1. Input safety
    input_check = filter_input(claim_text)
    if not input_check["safe"]:
        reason = (input_check.get("reason") or "").lower()
        health_keywords = ["health", "medical", "claim", "misleading", "harm", "vaccine", "cure", "treatment"]
        is_health = any(kw in reason for kw in health_keywords)
        if not is_health:
            return {"safe": False, "blocked_reason": input_check["reason"], "verdict": None}
        if debug:
            print("Nemotron flagged but health-related, proceeding:", reason[:100])

    # 2. Evidence retrieval
    evidence = _retrieve_evidence(claim_text)
    if debug:
        print(f"Retrieved {len(evidence)} sources from PubMed + Semantic Scholar")
        for e in evidence:
            print(f"  [{e['evidence_grade']}] {e['title'][:80]}")

    # 3. Build prompt in fine-tuning format
    user_prompt = _build_evidence_prompt(claim_text, evidence)
    messages = [
        {"role": "user", "content": f"[INST] {FEMMESTRAL_SYSTEM}\n\n{user_prompt} [/INST]"}
    ]
    inputs = femmestral_tokenizer(
        messages[0]["content"],
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(femmestral_model.device)

    # 4. Generate verdict
    with torch.no_grad():
        output_ids = femmestral_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=femmestral_tokenizer.pad_token_id,
        )
    response = femmestral_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if debug:
        print("\nFine-tuned model response:")
        print(response)

    # 5. Output safety
    output_check = filter_output(response)
    if not output_check["safe"]:
        return {"safe": False, "blocked_reason": output_check["reason"], "verdict": None}

    return {
        "safe": True,
        "claim": claim_text,
        "response": response,
        "sources": [{"title": p["title"], "url": p["url"], "grade": p["evidence_grade"]} for p in evidence],
        "source_count": len(evidence),
    }

print("femmestral_fact_check() ready -- traced in Weave")

femmestral_fact_check() ready -- traced in Weave


In [31]:
per_claim_results = []
for c in claims_out["claims"]:
    v = verify_claim_grounded(c["text"])
    v["trusted_citations"] = [u for u in v.get("citations", []) if is_trusted(u)]
    per_claim_results.append({"claim": c, "verification": v})

per_claim_results[:1]


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd153-28d4-7fd2-8bc5-db3ea11229db



STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
Doctors in Dubai are warning women:

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
The claim "Doctors in Dubai are warning women:" is incomplete. Without knowing what the doctors are warning women about, it is impossible to verify the claim.

**Stance suggestion:** Unclear

**Key evidence:**
*   The provided claim lacks specific content regarding the nature of the warning.
*   To verify any claim, the subject and predicate must be fully present.

**Source titles + URLs:**
No sources can be provided as the claim is incomplete and no search queries can be formulated.
STEP B

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd153-3522-78a3-8611-3470eddaabc7


STEP B OUTPUT RAW (should be JSON):
{"stance": "unclear", "confidence": "high", "rationale": "The claim \"Doctors in Dubai are warning women:\" is incomplete and lacks specific content regarding the nature of the warning. Without knowing what the doctors are warning women about, it is impossible to verify the claim.", "citations": []}
PARSED JSON OUTPUT (dict):
{
  "stance": "unclear",
  "confidence": "high",
  "rationale": "The claim \"Doctors in Dubai are warning women:\" is incomplete and lacks specific content regarding the nature of the warning. Without knowing what the doctors are warning women about, it is impossible to verify the claim.",
  "citations": []
}

STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd153-d9e6-73b8-a799-af4b5f830f6c


...

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd154-ecf8-7393-b946-453a2bb6a071


STEP B OUTPUT RAW (should be JSON):
{
  "stance": "unclear",
  "confidence": "high",
  "rationale": "The duration for which painkillers can be taken varies significantly based on the type of painkiller, the condition being treated, and individual health factors. Over-the-counter (OTC) painkillers for fever are generally recommended for up to 3 days, while those containing codeine should not exceed 3 days without medical advice. For general mild to moderate pain, OTC options like ibuprofen or acetaminophen should typically not be used for more than 7-10 days consecutively due to potential side effects. Before surgery, NSAIDs are often avoided for 5-7 days or up to two weeks, though acetaminophen is generally safe. After a COVID-19 vaccination, painkillers are advised post-shot if symptoms arise, not before. Opioid painkillers for acute severe pain are usually prescribed for a few days to weeks, with recommendations to stop within 5-7 days of hospital discharge, as long-term use is disco

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd154-fb5d-7c32-9baf-9e6eb338e8f4


STEP B OUTPUT RAW (should be JSON):
{
  "stance": "unclear",
  "confidence": "high",
  "rationale": "The claim to be verified was not provided in the input, therefore an assessment cannot be made.",
  "citations": []
}
PARSED JSON OUTPUT (dict):
{
  "stance": "unclear",
  "confidence": "high",
  "rationale": "The claim to be verified was not provided in the input, therefore an assessment cannot be made.",
  "citations": []
}

STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
Share this with every woman you know

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
**Stance Suggestion:** Unclear

**Key Evidence:**
*   The provide

[{'claim': {'claim_id': 'claim_1',
   'text': 'Doctors in Dubai are warning women:',
   'claim_type': 'authority_source',
   'check_priority': 'high',
   'entities': ['Doctors in Dubai']},
  'verification': {'stance': 'unclear',
   'confidence': 'high',
   'rationale': 'The claim "Doctors in Dubai are warning women:" is incomplete and lacks specific content regarding the nature of the warning. Without knowing what the doctors are warning women about, it is impossible to verify the claim.',
   'citations': [],
   'trusted_citations': []}}]

## Fine-tuned Model Verdicts (per claim)

Run the fine-tuned Mistral 7B through the full pipeline: Nemotron input safety, PubMed + Semantic Scholar evidence retrieval, fine-tuned model inference, Nemotron output safety.

In [32]:
# Run fine-tuned model on each extracted claim
finetuned_results = []
for c in claims_out["claims"]:
    claim_text = c.get("text", "").strip()
    if not claim_text:
        continue

    print("\n" + "=" * 70)
    print(f"CLAIM: {claim_text}")
    print(f"Type: {c.get('claim_type', 'unknown')} | Priority: {c.get('check_priority', 'unknown')}")
    print("-" * 70)

    result = femmestral_fact_check(claim_text, debug=True)
    finetuned_results.append({"claim": c, "femmestral_result": result})

print(f"\nProcessed {len(finetuned_results)} claims through the fine-tuned model")


CLAIM: Doctors in Dubai are warning women:
Type: authority_source | Priority: high
----------------------------------------------------------------------


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd155-0adc-7837-a76a-044cd3250925


Retrieved 2 sources from PubMed + Semantic Scholar
  [B] Pregnancy and the Use of Disease-Modifying Therapies in Patients with Multiple S
  [B] How do I: Evaluate the safety and legitimacy of unproven cellular therapies?


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd155-61fe-717a-9663-2c53b1b73c37



Fine-tuned model response:
**Verdict: Partially True**
**Confidence: Medium** | **Evidence: B** | **Severity: Medium**

The claim is partially true. Some disease-modifying therapies are not absolutely contraindicated during pregnancy, but many are recommended with caution or not prescribed during pregnancy due to potential risks. Interferons are not absolutely contraindicated during pregnancy.

**Share this correction:**
Hi! This is partially true. Some MS medications are not absolutely contraindicated during pregnancy, but many are recommended with caution. Interferons are not absolutely contraindicated during pregnancy. Speak to your doctor about your specific medication and pregnancy plans.

CLAIM: If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.
Type: medical_causal | Priority: high
----------------------------------------------------------------------
Retrieved 0 sources from PubMed + Semantic Scholar


...


Fine-tuned model response:
**Verdict: Partially True**
**Confidence: Medium** | **Evidence: A** | **Severity: Low**

Turmeric has anti-inflammatory properties due to curcumin. Drinking hot water with turmeric may help with minor inflammation. However, this is not a proven treatment for any medical condition.

**Source:** PubMed — Turmeric and Inflammation

**Share this correction:**
Turmeric may help with minor inflammation. Drinking hot water with turmeric is not a proven medical treatment.

CLAIM: Avoid painkillers for 3 days
Type: medical_advice | Priority: high
----------------------------------------------------------------------
Retrieved 5 sources from PubMed + Semantic Scholar
  [B] Diagnosis and Management of Headache: A Review.
  [B] Clinical management of cannabis withdrawal.
  [B] Hemorrhoidal Disease: A Review.
  [B] Medication-overuse headache: painkillers are not always the answer.
  [B] SARS-CoV-2: a potential trigger for subacute thyroiditis? Insights from a case r


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd156-34a2-7446-b699-23cdea1d9308



Fine-tuned model response:
**Verdict: Partially True**
**Confidence: Medium** | **Evidence: [4]** | **Severity: Medium**

**Share this correction:**
Painkillers can help manage acute headaches. However, overuse can lead to medication-overuse headache (MOH). MOH is a secondary headache disorder that develops from regular overuse of acute or symptomatic headache medication.

**Source:** Semantic Scholar (2020)

CLAIM: This is from a “hospital circular”.
Type: authority_source | Priority: high
----------------------------------------------------------------------
Retrieved 3 sources from PubMed + Semantic Scholar
  [B] Risk of Incident Psychosis and Mania With Prescription Amphetamines.
  [B] The oncomicropeptide APPLE promotes hematopoietic malignancy by enhancing transl
  [B] hsa_circ_0007919 induces LIG1 transcription by binding to FOXA1/TET1 to enhance 


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd156-7e8b-7b5b-9c69-4aa1588bee51



Fine-tuned model response:
**Verdict: Partially True**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Prescription amphetamines are associated with an increased risk of psychosis and mania, particularly at higher doses. This is a known side effect. However, the claim does not specify dose or duration of use.

**Share this correction:**
Prescription amphetamines can cause psychosis and mania. This risk is dose-dependent and increases with higher doses and longer use. Always discuss risks with your doctor.

CLAIM: Share this with every woman you know
Type: other | Priority: low
----------------------------------------------------------------------
Retrieved 2 sources from PubMed + Semantic Scholar
  [B] Police investigation and unethical "scientific interrogation".
  [B] Caring for baby's teeth starts before birth.

Fine-tuned model response:
**Verdict: Partially True**
**Confidence: Medium** | **Evidence: A** | **Severity: Medium**

The claim that caring for a baby's te

In [33]:
# Side-by-side comparison: Gemini grounding vs Fine-tuned model
print("=" * 90)
print("COMPARISON: Gemini Grounding vs Fine-tuned Femmestral 7B")
print("=" * 90)

for gemini_r, ft_r in zip(per_claim_results, finetuned_results):
    claim = gemini_r["claim"]["text"]
    gemini_stance = gemini_r["verification"].get("stance", "N/A")
    gemini_conf = gemini_r["verification"].get("confidence", "N/A")

    ft_response = ft_r["femmestral_result"].get("response", "")
    ft_sources = ft_r["femmestral_result"].get("source_count", 0)

    print(f"\nClaim: {claim}")
    print(f"  Gemini grounding:  stance={gemini_stance}, confidence={gemini_conf}")
    print(f"  Femmestral 7B:     {ft_response[:150]}...")
    print(f"  PubMed sources:    {ft_sources}")
    print("-" * 90)

COMPARISON: Gemini Grounding vs Fine-tuned Femmestral 7B

Claim: Doctors in Dubai are warning women:
  Gemini grounding:  stance=unclear, confidence=high
  Femmestral 7B:     **Verdict: Partially True**
**Confidence: Medium** | **Evidence: B** | **Severity: Medium**

The claim is partially true. Some disease-modifying thera...
  PubMed sources:    2
------------------------------------------------------------------------------------------

Claim: If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.
  Gemini grounding:  stance=contradicted, confidence=high
  Femmestral 7B:     **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Ibuprofen does not cause infertility. It is safe during the menstrual ...
  PubMed sources:    0
------------------------------------------------------------------------------------------

Claim: Drink hot water + turmeric
  Gemini grounding:  stance=supported, confidence=high
  Fe

In [34]:
@weave.op()
def text_pipeline_aggregate(per_claim: list) -> dict:
    contradicted = sum(1 for x in per_claim if x["verification"]["stance"] == "contradicted")
    unclear = sum(1 for x in per_claim if x["verification"]["stance"] == "unclear")

    if contradicted >= 1:
        return {"verdict": "misinformation", "confidence": "medium" if contradicted == 1 else "high"}
    if unclear >= max(1, len(per_claim)//2):
        return {"verdict": "unclear", "confidence": "medium"}
    return {"verdict": "likely_true", "confidence": "medium"}

# Keep old name as alias
aggregate_verdict = text_pipeline_aggregate

final = text_pipeline_aggregate(per_claim_results)
final

{'verdict': 'misinformation', 'confidence': 'medium'}

In [35]:
def compose_shareback(final: dict, per_claim: list) -> str:
    trusted = []
    for x in per_claim:
        trusted += x["verification"].get("trusted_citations", [])
    trusted = list(dict.fromkeys(trusted))[:2]  # unique, first 2

    if final["verdict"] == "misinformation":
        header = "Quick fact-check: this forward looks misleading."
    elif final["verdict"] == "unclear":
        header = "Quick fact-check: this forward can’t be verified confidently."
    else:
        header = "Quick fact-check: this forward appears broadly consistent with reputable sources."

    bullets = [f"• {x['claim']['text']} → {x['verification']['stance']}" for x in per_claim[:3]]
    src_line = ("Sources: " + " | ".join(trusted)) if trusted else ""
    tail = "If this is health-related, consider checking with a clinician."

    parts = [header] + bullets + ([src_line] if src_line else []) + [tail]
    return "\n".join(parts)

shareback = compose_shareback(final, per_claim_results)
print(shareback)


Quick fact-check: this forward looks misleading.
• Doctors in Dubai are warning women: → unclear
• If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility. → contradicted
• Drink hot water + turmeric → supported
If this is health-related, consider checking with a clinician.


In [36]:
stage8 = {
    "hashes": {"canonical_sha256": canonical_sha256, "raw_sha256": raw_sha256},
    "asset": {"path": str(msg_path), "type": "text"},
    "routing_model": MISTRAL_MODEL,
    "routing": routing,
    "claims": claims_out,
    "gemini_verification": per_claim_results,
    "femmestral_verification": finetuned_results,
    "final": final,
    "shareback": shareback,
}

stage8_path = OUTPUT_DIR / f"{canonical_sha256}_stage8_verdict.json"
stage8_path.write_text(json.dumps(stage8, indent=2, default=str), encoding="utf-8")
print("Saved:", stage8_path)

Saved: outputs/1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage8_verdict.json


In [37]:
# ===============================
# Cell R1: Realistic Reddit / Chat Messages
# ===============================

reddit_messages = [

"""Saw a post saying electric vehicles are actually worse for the environment
because battery production creates more pollution than gas cars ever will.
Feels like governments are hiding this.""",

"""My friend works in finance and told me Bitcoin transactions are totally
untraceable and governments can never track them.""",

"""Apparently NASA confirmed humans can see the Great Wall of China
from space without equipment. Crazy if true.""",

"""Someone told me WiFi routers consume massive electricity even when idle
and are one of the biggest causes of high electricity bills.""",

"""Read somewhere that AI will replace 90% of jobs within the next 5 years.
Seems inevitable honestly."""
]

print(" Loaded realistic Reddit-style messages")
print("Total messages:", len(reddit_messages))

 Loaded realistic Reddit-style messages
Total messages: 5


In [38]:
# ===============================
# Cell R2: Extract Claims
# ===============================

reddit_claim_sets = []

for msg in reddit_messages:

    print("\n====================================")
    print("REDDIT MESSAGE:")
    print(msg)

    claims = extract_claims_with_gemini(msg)

    print("\nEXTRACTED CLAIMS:")
    print(json.dumps(claims, indent=2))

    reddit_claim_sets.append(claims)

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd157-5bac-731f-b81a-9396b588fc5d



REDDIT MESSAGE:
Saw a post saying electric vehicles are actually worse for the environment
because battery production creates more pollution than gas cars ever will.
Feels like governments are hiding this.


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd157-7c29-7c74-bce5-f2f422d1b576



EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
      "claim_id": "EV_ENV_IMPACT_1",
      "text": "electric vehicles are actually worse for the environment",
      "claim_type": "statistical",
      "check_priority": "high",
      "entities": [
        "electric vehicles",
        "environment"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "BATTERY_POLLUTION_COMP_1",
      "text": "battery production creates more pollution than gas cars ever will",
      "claim_type": "statistical",
      "check_priority": "high",
      "entities": [
        "battery production",
        "pollution",
        "gas cars"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "GOV_CONCEALMENT_1",
      "text": "governments are hiding this",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "governments",
        "electric vehicles",
        "pollution",
        "environment"
      ],
      "implied_advice": null
    

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd157-89a7-72e4-a848-994232600759



EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
      "claim_id": "claim_1",
      "text": "My friend works in finance",
      "claim_type": "authority_source",
      "check_priority": "low",
      "entities": [
        "friend",
        "finance"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "claim_2",
      "text": "Bitcoin transactions are totally untraceable",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "Bitcoin transactions"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "claim_3",
      "text": "governments can never track them",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "governments",
        "Bitcoin transactions"
      ],
      "implied_advice": null
    }
  ]
}

REDDIT MESSAGE:
Apparently NASA confirmed humans can see the Great Wall of China
from space without equipment. Crazy if true.


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd157-9e3d-78b0-8f4b-9f57c175035e



EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
      "claim_id": "claim_1",
      "text": "NASA confirmed humans can see the Great Wall of China from space without equipment.",
      "claim_type": "authority_source",
      "check_priority": "high",
      "entities": [
        "NASA",
        "humans",
        "Great Wall of China",
        "space",
        "equipment"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "claim_2",
      "text": "humans can see the Great Wall of China from space without equipment.",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "humans",
        "Great Wall of China",
        "space",
        "equipment"
      ],
      "implied_advice": null
    }
  ]
}

REDDIT MESSAGE:
Someone told me WiFi routers consume massive electricity even when idle
and are one of the biggest causes of high electricity bills.


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd157-aa84-7c34-b96e-0e8f9b2cc66d



EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
      "claim_id": "claim_1",
      "text": "WiFi routers consume massive electricity even when idle.",
      "claim_type": "other",
      "check_priority": "medium",
      "entities": [
        "WiFi routers",
        "electricity"
      ]
    },
    {
      "claim_id": "claim_2",
      "text": "WiFi routers are one of the biggest causes of high electricity bills.",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "WiFi routers",
        "electricity bills"
      ]
    },
    {
      "claim_id": "claim_3",
      "text": "Someone told me these things.",
      "claim_type": "authority_source",
      "check_priority": "low",
      "entities": [
        "Someone"
      ],
      "implied_advice": null
    }
  ]
}

REDDIT MESSAGE:
Read somewhere that AI will replace 90% of jobs within the next 5 years.
Seems inevitable honestly.

EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
   

In [39]:
# ===============================
# Cell R3: Verify Claims
# ===============================

reddit_results = []

for claim_block in reddit_claim_sets:
    for c in claim_block["claims"]:

        print("\n------------------------------------")
        print("VERIFYING CLAIM:")
        print(c["text"])

        verification = verify_claim_grounded(c["text"])

        print("\nVERIFICATION OUTPUT:")
        print(json.dumps(verification, indent=2))

        reddit_results.append({
            "claim": c["text"],
            "verification": verification
        })

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd158-572e-7bbf-ab90-56b578cb4747



------------------------------------
VERIFYING CLAIM:
electric vehicles are actually worse for the environment

STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
electric vehicles are actually worse for the environment

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
Stance suggestion: Contradicted

Here's an evidence-based assessment of the claim:

*   **Overall Lifecycle Emissions:** Electric vehicles (EVs) consistently produce fewer greenhouse gas (GHG) emissions over their entire lifecycle compared to gasoline-powered cars, even when accounting for battery manufacturing and electricity generation. Studies show EVs can 

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd158-cecc-799c-b638-03f8db4ec46e


STEP B OUTPUT RAW (should be JSON):
{
  "stance": "contradicted",
  "confidence": "high",
  "rationale": "While electric vehicles (EVs) have higher manufacturing emissions due to battery production, they produce significantly fewer greenhouse gas emissions over their entire lifecycle compared to gasoline cars. EVs have zero tailpipe emissions and typically reach a break-even point in total emissions within 1.5-2 years of driving. The environmental benefits are present even when charged on the average U.S. power grid, and battery recycling technologies are improving to address end-of-life concerns.",
  "citations": [
    "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGKUpmFDPvg0TuXbfJsgrEPBP7lJkfajDMmDCpi7lypOBG53aRgJj-Bb9neKnPYo9hk18jvi_oznUNmzrKgldmFLYt1Q532hv-PQhcxfPRw9P7Ruvbk-84deGtpfkmc11Qf_J2BDtyhw2GrfGSURICY0DVCzrucRi7J0t0zaHQ7UxkNNpRJndVNDllZ1wpDnQZENTDIIyYSJroxcw==",
    "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHbIR8yH6HapkXPc

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd159-3b7d-7787-95ac-64d3432574ba


...

STEP B OUTPUT RAW (should be JSON):
{"stance": "unclear", "confidence": "low", "rationale": "The claim cannot be verified as no specific claim was provided in the input.", "citations": []}
PARSED JSON OUTPUT (dict):
{
  "stance": "unclear",
  "confidence": "low",
  "rationale": "The claim cannot be verified as no specific claim was provided in the input.",
  "citations": []
}

VERIFICATION OUTPUT:
{
  "stance": "unclear",
  "confidence": "low",
  "rationale": "The claim cannot be verified as no specific claim was provided in the input.",
  "citations": []
}

------------------------------------
VERIFYING CLAIM:
AI will replace 90% of jobs within the next 5 years.

STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
AI will replace 90% of jobs within the next 5 years.

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd15b-b57e-7884-8912-fc8f234330be


STEP B OUTPUT RAW (should be JSON):
{"stance": "contradicted", "confidence": "high", "rationale": "The claim that AI will replace 90% of jobs within the next 5 years is contradicted by current expert predictions. Most studies project a much lower percentage of job displacement, typically between 25% to 30% by 2030, not 90%. The World Economic Forum estimates a net gain of 78 million jobs globally by 2030, with 170 million new roles created against 92 million displaced. Goldman Sachs Research projects a mild and short-lived impact on overall employment, with new job opportunities expected to emerge. While some experts suggest AI could eliminate up to 50% of *entry-level white-collar jobs*, this is a specific segment and not representative of the entire job market. AI's impact is more often described as transforming or augmenting existing jobs by automating specific tasks, rather than eliminating entire occupations, and there is a growing demand for new AI-specific and human-centric skil

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral-text-pipeline/r/call/019cd15c-3a7e-74c3-832f-2f99ebea4c67


STEP B OUTPUT RAW (should be JSON):
{
  "stance": "contradicted",
  "confidence": "high",
  "rationale": "The claim that AI will replace 90% of jobs within the next 5 years is largely contradicted by current expert predictions and research. Goldman Sachs estimates AI could displace 6-7% of the US workforce, with this impact likely transitory and over a longer timeframe. The World Economic Forum projected net job creation by 2025 and 2030, with more jobs created than displaced. Forrester predicts 6.1% of US jobs lost by 2030, while 20% will be influenced rather than replaced. Other estimates suggest up to 30% of jobs could be automated by the mid-2030s, with 60% significantly altered. While some experts, like Anthropic CEO, have warned of higher displacement in specific sectors (e.g., 50% of entry-level white-collar jobs within five years), this is not representative of all jobs. Many sources emphasize that AI's impact will involve automating tasks within jobs, requiring workers to adap

In [40]:
# Summary of Reddit claim verification results
for r in reddit_results:
    print("\n====================================")
    print("CLAIM:", r["claim"])
    print("STANCE:", r["verification"].get("stance", "N/A"))
    print("CONFIDENCE:", r["verification"].get("confidence", "N/A"))

    print("\nTOP SOURCES:")
    for url in r["verification"].get("citations", [])[:3]:
        print(" -", url)

    print("RATIONALE:", r["verification"].get("rationale", "N/A")[:200])


CLAIM: electric vehicles are actually worse for the environment
STANCE: contradicted
CONFIDENCE: high

TOP SOURCES:
 - https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGKUpmFDPvg0TuXbfJsgrEPBP7lJkfajDMmDCpi7lypOBG53aRgJj-Bb9neKnPYo9hk18jvi_oznUNmzrKgldmFLYt1Q532hv-PQhcxfPRw9P7Ruvbk-84deGtpfkmc11Qf_J2BDtyhw2GrfGSURICY0DVCzrucRi7J0t0zaHQ7UxkNNpRJndVNDllZ1wpDnQZENTDIIyYSJroxcw==
 - https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHbIR8yH6HapkXPcLcivoxxA7nt9GsP11QonbdGSp9k502hFIguUMpTQPIgfvCcVKZx6PPEkkxfLVArFjeuxqNvfrUcCn5pYHr2kbgY7g0HJJTFQaSvDzdrTJTI0q43nT_bQGJFDZcrIWl2SQ0SEkRuHlr8VHoWcrtQnkE-TBXVNBjeATAn820bf4GVKt6eB3iPRA2khhXO
 - https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo6PqE16YQ_dzcmT26OLbgSXcrURNaHYvvm_L2Ds9meq-KWA1mP6rjQCDHWqBPatTy136FNQ38EFBYEK92D6ZmYqBV-k6kLXObqBo_l7tOIIk-wPmgT4-FFk-TqvHvX-MFkKJTGaCbLd_YAFIZMZjJ3KJ0RxUJ1I-7s2IZFvx6ekt8c7f_uvVVRg==
RATIONALE: While electric vehicles (EVs) have higher manufacturing

In [41]:
  from google.colab import drive
  drive.mount('/content/drive')

  import shutil
  from pathlib import Path

  drive_dir = Path("/content/drive/MyDrive/femmestral/outputs")
  drive_dir.mkdir(parents=True, exist_ok=True)

  import glob
  for f in glob.glob("outputs/*.json"):
      shutil.copy(f, drive_dir / Path(f).name)
      print("Saved:", Path(f).name)

  print("All outputs saved to Google Drive: femmestral/outputs/")

Mounted at /content/drive
Saved: 1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage5.json
Saved: 1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage8_verdict.json
Saved: 1ff9a033def0309675ba39a20e3dfcd02d76f3b2efc54353501852582fbef362_stage6_claims.json
All outputs saved to Google Drive: femmestral/outputs/


fatal: destination path '/content/femmestral-repo' already exists and is not an empty directory.
cp: cannot stat '/content/WomenFalseInformation_full_fixed.ipynb': No such file or directory
/content/femmestral-repo
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://HF_TOKEN_REDACTED@github.com': No such device or address
